# 09 · Model B: hierarchical Gaussian processes, and whether amplifier labels are needed

*Level: advanced. Guide Section 13. Code: `structured.HierGP`, `PhysicsStructured(kind="hier")`.*

**Idea.** Amplifiers differ but are related. A hierarchical model learns:
* what they share;
* how much each deviates.

A new module then needs a few packets, not a whole scan.

In [ ]:
from tutorial_utils import *

from ml import structured as st
from scipy.stats import spearmanr

## 1. Shrinkage: borrowing strength

An amplifier's offset $\delta$ is drawn with spread $c_a$. With $k$ of its points (noise $\sigma_n$), averaging $\bar r$, the best estimate is

$$\hat\delta = \frac{c_a^2}{c_a^2 + \sigma_n^2/k}\,\bar r ,$$

which is pulled towards 0 when there are few points. A simulation compares it with the raw mean.

In [ ]:
rng = np.random.default_rng(0); c_a, s_n = 0.3, 0.5
for k in (1, 2, 5, 20):
    delta = rng.normal(0, c_a, 20000); rbar = delta + rng.normal(0, s_n / np.sqrt(k), 20000)
    shrunk = c_a ** 2 / (c_a ** 2 + s_n ** 2 / k) * rbar
    print(f"k = {k:2d}: RMS error raw mean {np.sqrt(np.mean((rbar - delta) ** 2)):.3f}   shrunk {np.sqrt(np.mean((shrunk - delta) ** 2)):.3f}")

## 2. The kernel: sums of independent pieces

For amplifier $j$ on module $m$: $f_j(x) = f(x) + \delta_m(x) + \delta_j(x)$. Independent GPs add their kernels:

$$k(i,i') = \sigma_f^2 M + \mathbb 1[m=m'](c_m^2 + \sigma_m^2 M) + \mathbb 1[j=j'](c_a^2 + \sigma_a^2 M) + \sigma_n^2\mathbb 1[i=i'] .$$

* $M$ is the Matérn correlation.
* The indicators switch a term on only for pairs on the same module or the same amplifier.

In [ ]:
x = np.tile(np.linspace(0, 5, 10), 3); amp = np.repeat([0, 1, 2], 10)
M = (lambda r: (1 + np.sqrt(5) * r + 5 / 3 * r ** 2) * np.exp(-np.sqrt(5) * r))(np.abs(x[:, None] - x[None, :]) / 1.5)
K = 1.0 * M + (amp[:, None] == amp[None, :]) * (0.5 + 0.1 * M)
plt.figure(figsize=(4.2, 3.6)); plt.imshow(K); plt.colorbar(); plt.title("3 amplifiers × 10 settings: shared + same-amplifier blocks"); plt.show()

## 3. A new amplifier with 0 and 1 of its own points

`HierGP(levels=("amp",))` learns all the amplitudes from the data by maximizing the marginal likelihood.

In [ ]:
shared = lambda x: np.sin(x) + 0.2 * x; offsets = [0.9, -0.6, 0.2, 1.4]
X, A_, Yv = [], [], []
for j in range(3):
    xx = np.sort(rng.uniform(0, 8, 9)); X.append(xx); A_.append(np.full(9, j)); Yv.append(shared(xx) + offsets[j] + 0.1 * rng.normal(size=9))
X, A_, Yv = map(np.concatenate, (X, A_, Yv)); xs = np.linspace(0, 8, 200)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), sharey=True)
for a_, k in zip(ax, (0, 1)):
    Xk, Ak, Yk = (np.append(X, 5.0), np.append(A_, 3), np.append(Yv, shared(5.0) + offsets[3])) if k else (X, A_, Yv)
    gp = st.HierGP(levels=("amp",)).fit({"x": Xk[:, None], "amp": Ak}, Yk)
    for j in range(4):
        m, s = gp.predict({"x": xs[:, None], "amp": np.full(200, j)})
        a_.plot(xs, m[:, 0], color=f"C{j}", lw=2.5 if j == 3 else 1)
        if j == 3: a_.fill_between(xs, m[:, 0] - 2 * s[:, 0], m[:, 0] + 2 * s[:, 0], color="C3", alpha=0.15)
    a_.plot(xs, shared(xs) + offsets[3], "k:"); a_.scatter(X, Yv, c=[f"C{j}" for j in A_], s=10)
    a_.set_title(f"new amplifier (red) with {k} of its own points")
plt.tight_layout(); plt.show()
print("fitted amplitudes (in units of the target sd):", {k: round(v, 3) for k, v in gp.hyperparameters().items()})

## 4. Do the models need amplifier labels? A new module on the toy CCD

Training uses a whole scan of module 100 plus $k$ packets of module 101; testing uses new packets of module 101. Two models are compared:
* **A+B**: told the module and amplifier (the hierarchical levels);
* **A, no labels**: each amplifier's own traces instead (the default).

Besides the error, two scores matter for optimization:
* **shape**: the error after removing each amplifier's mean error;
* **rank**: the within-amplifier Spearman correlation, i.e. whether the settings are put in the right order.

In [ ]:
def scores(df_, m):
    t = np.log(df_["truth.sigma_e"].to_numpy()); r = m[:, 0] - t
    amp = (df_.board.astype(str) + df_.channel.astype(str)).to_numpy()
    shape = r.copy()
    for a in np.unique(amp): shape[amp == a] -= r[amp == a].mean()
    rank = np.mean([spearmanr(m[amp == a, 0], t[amp == a])[0] for a in np.unique(amp)])
    return 100 * np.sqrt(np.mean(r ** 2)), 100 * np.sqrt(np.mean(shape ** 2)), rank

tr, ctr = synthetic.make_dataset(80, seed=1, n_boards=2); te, cte = synthetic.make_dataset(40, seed=99, n_boards=2)
te_new = (te.board == 101).to_numpy(); te, cte = te[te_new].reset_index(drop=True), {k: v[te_new] for k, v in cte.items()}
pk = tr.packet_id.str[4:].astype(int)
for k in (0, 2):
    keep = ((tr.board == 100) | (pk < k)).to_numpy()
    d, c = tr[keep].reset_index(drop=True), {kk: v[keep] for kk, v in ctr.items()}
    Yd = data.make_targets(d)[0]
    for label, kind in (("A+B, amplifier/module labels", "hier"), ("A, no labels (own traces)", "none")):
        prep = st.StructuredPrep().fit(d)
        m, s = st.PhysicsStructured(kind).fit(prep.transform(d, c), Yd).predict(prep.transform(te, cte))
        e, sh, rk = scores(te, m)
        print(f"new module, {k} of its packets | {label:30s}: σe error {e:5.1f} %, shape {sh:4.1f} %, rank {rk:.2f}")

With 0 packets of the new module, every model is off in *level* (its gains are unknown), but the shape and the order of the settings are right.

After 2 packets both are within a few percent. In this single run the labelled model is ahead; averaged over 3 training scans in the project's
test (`benchmark_amp.py`) they were 2.0 % against 2.2 %. Both put the settings in the right order, and the label-free model works for **any**
amplifier. That is why the default uses no labels.

## When to use it

* **Good:**
  * several amplifiers or modules;
  * a new module with 1–2 packets;
  * honest error bars for an amplifier never seen.
* **Not needed:** when each amplifier's own traces are inputs. The module level only means something with ≥ 2 modules (model G, several modules, is this GP fitted across modules).

Next: **10 · Dark current and CIC, the joint fit.**